# 01 — Data Prep (D1–D2)

**VinDr-CXR → YOLO format.** Positive-only subset, WBF rater fusion, stratified 80/10/10.

> ⚠️ **This notebook is where the paper quietly dies.** If rater fusion is wrong, every downstream number is wrong, and the D4 sanity gate will *not* catch it — broken-but-plausible labels give a broken-but-plausible 0.25 mAP that looks fine.
>
> **Do not skip the visual check in §5.** Look at 20 images before training anything.

**Settings:** accelerator can be `None` for this notebook (CPU only), internet ON.

**Gate:** §7 `verify()` must pass before starting notebook 02.

In [ ]:
!pip install -q ensemble-boxes

import sys
# Attach the repo as a Kaggle Dataset, or clone it. Adjust to taste.
REPO = "/kaggle/input/vindr-iccit-repo"   # <-- path to this repo on Kaggle
sys.path.insert(0, REPO)

from pathlib import Path
import pandas as pd, numpy as np

from src import config as C
from src.data import fusion, prepare

print(C.as_dict())

## 1. Load raw annotations

Uses the pre-resized 1024px JPG dataset. **Do not re-derive from the 192 GB DICOM set** — hours of compute for no benefit.

In [ ]:
IMAGE_DIR = C.RAW_IMAGES / "train"   # adjust if the dataset lays out differently

df_raw = prepare.load_raw(C.TRAIN_CSV)
df_raw.head()

In [ ]:
# Sanity: rater distribution. Expect R8/R9/R10 to dominate positive findings
# and R1-R7 to be near-zero. If this does NOT hold, you have a different
# release than the docs assume -- stop and investigate.
if "rad_id" in df_raw.columns:
    print(df_raw[df_raw.class_id != 14].rad_id.value_counts())

## 2. Positive-only subset

~4.4K of 15K images. The compute enabler — ~3.4× epoch-time reduction. Established protocol on this dataset (`sunghyunjun` trained its detector on abnormal images only).

**Report this in the paper.** It means the model never sees normal images and cannot be evaluated on normal-vs-abnormal discrimination.

In [ ]:
df_pos = prepare.positive_only(df_raw)
dims = prepare.get_image_dims(IMAGE_DIR, df_pos.image_id.unique())
print(f"{len(dims)} images measured")

## 3. Multi-rater fusion

3 radiologists per image → single training labels. WBF @ IoU 0.5.

**Unsettled, state it in the paper:** the `sunghyunjun` reference's own ablation has WBF *winning* on private LB (0.185 vs 0.181) while *losing* on CV (0.4158 vs 0.4419). The author chose NMS by reasoning about test-set labeling, not measurement. Don't inherit either as established.

In [ ]:
df_fused = fusion.fuse_dataset(
    df_pos, dims, method=C.FUSION_METHOD, iou_thr=C.FUSION_IOU
)
df_fused.head()

## 4. Fusion report — sanity check #1

**Red flags:**
- retention ≈100% → fusion did nothing, `iou_thr` too high
- retention ≈33% → collapsing all 3 raters always, `iou_thr` too low
- any class → 0 → bug

Expect roughly **40–70%** overall.

In [ ]:
rep = fusion.fusion_report(df_pos, df_fused)
rep

## 5. Visual check — sanity check #2 (MANDATORY)

Red dotted = individual rater boxes. Green = fused.

**What you are looking for:**
- fused boxes sit sensibly inside the rater cluster
- no box spanning the whole image (over-merging)
- no duplicate near-identical fused boxes (under-merging)
- anatomically plausible (cardiomegaly over the heart, etc.)

**If these look wrong, stop.** Everything downstream inherits the error.

In [ ]:
rng = np.random.default_rng(C.SEED)
sample_ids = list(rng.choice(df_fused.image_id.unique(), size=20, replace=False))

fig = fusion.plot_fusion_examples(df_pos, df_fused, IMAGE_DIR, sample_ids[:6])

In [ ]:
fig = fusion.plot_fusion_examples(df_pos, df_fused, IMAGE_DIR, sample_ids[6:12])

## 6. Stratified split + YOLO export

Stratified on each image's **rarest** present class — protects tail classes (Pneumothorax, Other lesion) from vanishing out of val/test, which would make their per-class AP undefined and break the Axis-A analysis.

In [ ]:
splits = prepare.stratified_split(df_fused, split=C.SPLIT, seed=C.SEED)
prepare.class_distribution(df_fused, splits)

In [ ]:
C.DATA_ROOT.mkdir(parents=True, exist_ok=True)
prepare.to_yolo_labels(df_fused, dims, splits, C.DATA_ROOT, IMAGE_DIR)
data_yaml = prepare.write_data_yaml(C.DATA_ROOT, C.CLASSES)

## 7. Verify — GATE

Must print `✓ verify passed` before notebook 02.

In [ ]:
ok = prepare.verify(C.DATA_ROOT, splits)
assert ok, "data prep failed verification -- do NOT proceed to training"

df_fused.to_csv(C.DATA_ROOT / "fused_boxes.csv", index=False)
pd.Series(splits).to_json(C.DATA_ROOT / "splits.json")
print(f"\ndone. data.yaml -> {data_yaml}")

## 8. Save as Kaggle Dataset

**Do this.** `/kaggle/working` dies with the 12h session, and rebuilding the dataset costs an hour you won't have on D5.

`File → Save Version → Save Output`, then attach the output as an input dataset in notebook 02.

Keep it **private** — VinDr-CXR carries a data use agreement restricting redistribution.